# create a clm nc and fill it with ucla-roms output

In [2]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from cmocean import cm
import os
import netCDF4 as nc
import glob
from tqdm import tqdm


In [14]:
import netCDF4
from datetime import datetime
import os

def make_clm_from_cdl(his_file, clm_file):

    his = netCDF4.Dataset(his_file)
    clm = netCDF4.Dataset(clm_file, 'w', format='NETCDF4')

    # ---------- Global attributes ----------
    clm.Description = "CLimatology for offline simulation"
    clm.Author = "Siming"
    clm.Created = "2026-04-16T10:00:00"
    clm.type = "ROMS CLM file"

    # ---------- Dimensions ----------
    xi_rho = 602
    xi_u   = 601
    xi_v   = 602
    xi_psi = 601
    
    eta_rho = 402
    eta_u   = 402
    eta_v   = 401
    eta_psi = 401

    s_rho = 45
    s_w   = 46

    ntime = 744
    
    dims_size = {
    'xi_rho': xi_rho,
    'xi_u': xi_u,
    'xi_v': xi_v,
    'xi_psi': xi_psi,
    'eta_rho': eta_rho,
    'eta_u': eta_u,
    'eta_v': eta_v,
    'eta_psi': eta_psi,
    's_rho': s_rho,
    's_w': s_w,
    'ocean_time': ntime
}

    clm.createDimension('xi_rho', xi_rho)
    clm.createDimension('xi_u', xi_u)
    clm.createDimension('xi_v', xi_v)
    clm.createDimension('xi_psi', xi_psi)

    clm.createDimension('eta_rho', eta_rho)
    clm.createDimension('eta_u', eta_u)
    clm.createDimension('eta_v', eta_v)
    clm.createDimension('eta_psi', eta_psi)

    clm.createDimension('N', s_rho)
    clm.createDimension('s_rho', s_rho)
    clm.createDimension('s_w', s_w)

    clm.createDimension('tracer', 2)
    clm.createDimension('boundary', 4)

    clm.createDimension('ocean_time', ntime)
    clm.createDimension('zeta_time', ntime)
    clm.createDimension('u2d_time', ntime)
    clm.createDimension('v2d_time', ntime)
    clm.createDimension('u3d_time', ntime)
    clm.createDimension('v3d_time', ntime)
    clm.createDimension('temp_time', ntime)
    clm.createDimension('salt_time', ntime)
    clm.createDimension('srf_time', ntime)
    clm.createDimension('ssf_time', ntime)

    # ---------- Time variables ----------
    def add_time_var(name):
        v = clm.createVariable(name, 'f8', (name,))
        v.units = "seconds since 0001-01-01 00:00:00"
        v.calendar = "gregorian_proleptic"
        return v

    zeta_time = add_time_var('zeta_time')
    zeta_time.long_name = "time for free-surface"
    zeta_time.field = "zeta_time, scalar, series"

    u2d_time = add_time_var('u2d_time')
    u2d_time.long_name = "time for vertically integrated u-momentum component"
    u2d_time.field = "u2d_time, scalar, series"

    v2d_time = add_time_var('v2d_time')
    v2d_time.long_name = "time for vertically integrated v-momentum component"
    v2d_time.field = "v2d_time, scalar, series"

    u3d_time = add_time_var('u3d_time')
    u3d_time.long_name = "time for u-momentum component"
    u3d_time.field = "u3d_time, scalar, series"

    v3d_time = add_time_var('v3d_time')
    v3d_time.long_name = "time for v-momentum component"
    v3d_time.field = "v3d_time, scalar, series"

    ocean_time = add_time_var('ocean_time')
    ocean_time.long_name = "time for S-coordinate vertical momentum component"
    ocean_time.field = "ocean_time, scalar, series"

    temp_time = add_time_var('temp_time')
    temp_time.long_name = "time for potential temperature"
    temp_time.field = "temp_time, scalar, series"

    salt_time = add_time_var('salt_time')
    salt_time.long_name = "time for salinity"
    salt_time.field = "salt_time, scalar, series"

    srf_time = add_time_var('srf_time')
    srf_time.long_name = "time for solar shortwave radiation flux"
    srf_time.field = "srf_time, scalar, series"

    ssf_time = add_time_var('ssf_time')
    ssf_time.long_name = "time for surface net heat flux"
    ssf_time.field = "ssf_time, scalar, series"

    # ---------- Write time values ----------
    ocean_time[:] = his.variables['ocean_time'][:]
    zeta_time[:] = his.variables['ocean_time'][:]
    u2d_time[:] = his.variables['ocean_time'][:]
    v2d_time[:] = his.variables['ocean_time'][:]
    u3d_time[:] = his.variables['ocean_time'][:]
    v3d_time[:] = his.variables['ocean_time'][:]
    temp_time[:] = his.variables['ocean_time'][:]
    salt_time[:] = his.variables['ocean_time'][:]
    srf_time[:] = his.variables['ocean_time'][:]
    ssf_time[:] = his.variables['ocean_time'][:]

    # ---------- Physical variables (streaming write) ----------
    def add_var(name, dims):
        # 把字符串维度名转换成对应的维度大小
        chunk_sizes = [1]  # 时间维
        for d in dims[1:]:
            chunk_sizes.append(dims_size[d])

        return clm.createVariable(
            name, 'f8', dims,
            chunksizes=tuple(chunk_sizes),
            zlib=True,
            complevel=1
        )

    zeta = add_var('zeta', ('ocean_time', 'eta_rho', 'xi_rho'))
    zeta.long_name = "free-surface"
    zeta.units = "meter"
    zeta.field = "free-surface, scalar, series"
    zeta.time = "zeta_time"

    ubar = add_var('ubar', ('ocean_time', 'eta_u', 'xi_u'))
    ubar.long_name = "vertically integrated u-momentum component"
    ubar.units = "meter second-1"
    ubar.field = "ubar-velocity, scalar, series"
    ubar.time = "u2d_time"

    vbar = add_var('vbar', ('ocean_time', 'eta_v', 'xi_v'))
    vbar.long_name = "vertically integrated v-momentum component"
    vbar.units = "meter second-1"
    vbar.field = "vbar-velocity, scalar, series"
    vbar.time = "v2d_time"

    u = add_var('u', ('ocean_time', 's_rho', 'eta_u', 'xi_u'))
    u.long_name = "u-momentum component"
    u.units = "meter second-1"
    u.field = "u-velocity, scalar, series"
    u.time = "u3d_time"

    v = add_var('v', ('ocean_time', 's_rho', 'eta_v', 'xi_v'))
    v.long_name = "v-momentum component"
    v.units = "meter second-1"
    v.field = "v-velocity, scalar, series"
    v.time = "v3d_time"

    omega = add_var('omega', ('ocean_time', 's_w', 'eta_rho', 'xi_rho'))
    omega.long_name = "S-coordinate vertical momentum component"
    omega.units = "meter second-1"
    omega.field = "omega, scalar, series"
    omega.time = "ocean_time"

    temp = add_var('temp', ('ocean_time', 's_rho', 'eta_rho', 'xi_rho'))
    temp.long_name = "potential temperature"
    temp.units = "Celsius"
    temp.field = "temperature, scalar, series"
    temp.time = "temp_time"

    salt = add_var('salt', ('ocean_time', 's_rho', 'eta_rho', 'xi_rho'))
    salt.long_name = "salinity"
    salt.field = "salinity, scalar, series"
    salt.time = "salt_time"

    swrad = add_var('swrad', ('ocean_time', 'eta_rho', 'xi_rho'))
    swrad.long_name = "solar shortwave radiation flux"
    swrad.units = "watt meter-2"
    swrad.field = "shortwave radiation, scalar, series"
    swrad.time = "srf_time"

    shflux = add_var('shflux', ('ocean_time', 'eta_rho', 'xi_rho'))
    shflux.long_name = "surface net heat flux"
    shflux.units = "watt meter-2"
    shflux.field = "surface heat flux, scalar, series"
    shflux.time = "ssf_time"

    # ---------- Streaming write ----------
    for t in tqdm(range(ntime)):
        if t % 50 == 0:
            print(f'Writing time step {t}/{ntime}')

        zeta[t] = his.variables['zeta'][t]
        ubar[t] = his.variables['ubar'][t]
        vbar[t] = his.variables['vbar'][t]
        u[t] = his.variables['u'][t]
        v[t] = his.variables['v'][t]
        omega[t] = his.variables['omega'][t]
        temp[t] = his.variables['temp'][t]
        salt[t] = his.variables['salt'][t]

        # ⚠️ 如果你的输入文件没有 swrad / shflux，用 0 填充
        if 'swrad' in his.variables:
            swrad[t] = his.variables['swrad'][t]
        else:
            swrad[t] = 0.0

        if 'shflux' in his.variables:
            shflux[t] = his.variables['shflux'][t]
        else:
            shflux[t] = 0.0

    clm.close()
    his.close()
    print('✅ CLM file created successfully')

if __name__ == '__main__':
    make_clm_from_cdl(
        his_file='/leader/user/zsm/TWS2_L1_ucla_new/TWS_L1_his_201903_full.nc',
        clm_file='/leader/user/zsm/twc_clm.nc'
    )

  0%|                                                                                                                  | 0/744 [00:00<?, ?it/s]

Writing time step 0/744


  7%|██████▉                                                                                                | 50/744 [07:34<1:49:03,  9.43s/it]

Writing time step 50/744


 13%|█████████████▋                                                                                        | 100/744 [15:10<1:36:49,  9.02s/it]

...

Writing time step 650/744


 94%|███████████████████████████████████████████████████████████████████████████████████████████████▉      | 700/744 [1:40:28<05:47,  7.90s/it]

Writing time step 700/744


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [1:46:47<00:00,  8.61s/it]


✅ CLM file created successfully


In [2]:
era_path='/pacific1/user/biyesheng_zsm/TWS_L1_input/TWS_era2019010203.nc'
era=xr.open_dataset(era_path)


/tmp/ipykernel_3377578/1102898067.py:2: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  era=xr.open_dataset(era_path)


In [3]:
era

<xarray.Dataset> Size: 14GB
Dimensions:    (time: 2137, eta_rho: 402, xi_rho: 602, rad_time: 2137)
Coordinates:
  * time       (time) timedelta64[ns] 17kB 6940 days 00:00:00 ... 7029 days 0...
  * rad_time   (rad_time) timedelta64[ns] 17kB 6940 days 00:00:00 ... 7029 da...
Dimensions without coordinates: eta_rho, xi_rho
Data variables:
    rain       (time, eta_rho, xi_rho) float32 2GB ...
    uwnd       (time, eta_rho, xi_rho) float32 2GB ...
    vwnd       (time, eta_rho, xi_rho) float32 2GB ...
    wnd_time   (time) timedelta64[ns] 17kB ...
    Tair       (time, eta_rho, xi_rho) float32 2GB ...
    qair       (time, eta_rho, xi_rho) float32 2GB ...
    tair_time  (time) timedelta64[ns] 17kB ...
    qair_time  (time) timedelta64[ns] 17kB ...
    swrad      (rad_time, eta_rho, xi_rho) float32 2GB ...
    lwrad      (rad_time, eta_rho, xi_rho) float32 2GB ...
Attributes:
    title:                 ROMS surface forcing file created by ROMS-Tools
    roms_tools_version:    3.0.1.dev7+gbba77f9
    start_time:            2019-01-01 00:00:00
    end_time:              2019-03-31 00:00:00
    source:                ERA5
    correct_radiation:     True
    wind_dropoff:          True
    use_coarse_grid:       False
    model_reference_date:  2000-01-01 00:00:00
    type:                  physics

In [4]:
swrad=era['swrad'].values[1392:-1,:,:]

In [5]:
swrad.shape

(744, 402, 602)

In [6]:
flx_path='/leader/user/zsm/TWS2_L1_ucla_new/TWS_L1_flx_full.nc'
flx=xr.open_dataset(flx_path)
flx

<xarray.Dataset> Size: 3GB
Dimensions:     (time: 744, eta_rho: 402, xi_u: 601, eta_v: 401, xi_rho: 602)
Coordinates:
  * time        (time) float64 6kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
Dimensions without coordinates: eta_rho, xi_u, eta_v, xi_rho
Data variables:
    ocean_time  (time) float64 6kB ...
    sustr       (time, eta_rho, xi_u) float32 719MB ...
    svstr       (time, eta_v, xi_rho) float32 718MB ...
    shflx       (time, eta_rho, xi_rho) float32 720MB ...
    ssflx       (time, eta_rho, xi_rho) float32 720MB ...
Attributes: (12/48)
    CDI:                Climate Data Interface version 2.4.4 (https://mpimet....
    Conventions:        CF-1.6
    global_x:           600
    global_y:           400
    title:              Test for Iceland.
    grid_file:          /pacific1/user/biyesheng_zsm/TWS_L1_input/TWS2_L1_gri...
    ...                 ...
    pipe_frc_options:   OFF
    particle_options:   OFF
    git_version:        bfa4c0a5b663cd7b44f06647f2f3e6313d76c7ac
    type:               surface flux                              history
    history:            Thu Apr 16 21:40:34 2026: cdo mergetime TWS_L1_flx_hi...
    CDO:                Climate Data Operators version 2.4.4 (https://mpimet....

In [7]:
clm1=xr.open_dataset('/leader/user/zsm/TWS2_L1_ucla_new/TWS_L1_his_201903_full.nc')
SSS=clm1['salt'].values[:,-1,:,:]

In [8]:
SSS.shape

(744, 402, 602)

In [9]:
sustr=flx['sustr'].values
svstr=flx['svstr'].values
shflux=flx['shflx'].values*3985*1027.5
swflux=flx['ssflx'].values/SSS

/tmp/ipykernel_3377578/541526754.py:4: RuntimeWarning: invalid value encountered in divide
  swflux=flx['ssflx'].values/SSS


In [28]:
# clm=nc.Dataset('/leader/user/zsm/twc_clm.nc')

In [34]:
# clm.close()

In [35]:
clm = netCDF4.Dataset(
    '/leader/user/zsm/twc_clm.nc',
    'a'   # 'a' = append / write
)

shflux_var = clm.variables['shflux']
swrad_var = clm.variables['swrad']


ntime = shflux.shape[0]

for t in range(ntime):
    if t % 50 == 0:
        print(f"Writing shflux time {t}/{ntime}")

    shflux_var[t, :, :] = shflux[t, :, :].astype(np.float64)
    swrad_var[t, :, :] = swrad[t, :, :].astype(np.float64)


clm.close()

print("shflux written and CLM file saved successfully")

Writing shflux time 0/744
Writing shflux time 50/744
Writing shflux time 100/744
Writing shflux time 150/744
Writing shflux time 200/744
Writing shflux time 250/744
Writing shflux time 300/744
Writing shflux time 350/744
Writing shflux time 400/744
Writing shflux time 450/744
Writing shflux time 500/744
Writing shflux time 550/744
Writing shflux time 600/744
Writing shflux time 650/744
Writing shflux time 700/744
shflux written and CLM file saved successfully


# create forcing file

In [10]:
import netCDF4
import numpy as np
from datetime import datetime

def make_frc_shell(frc_file, his_file):

    # ---------- 打开历史文件以获取时间和维度 ----------
    his = netCDF4.Dataset(his_file)
    ntime = his.dimensions['time'].size

    clm = netCDF4.Dataset(frc_file, 'w', format='NETCDF4')

    # ---------- Global attributes ----------
    clm.Description = "Forcing for offline simulation"
    clm.Created = datetime.now().strftime("2026-04-16T%H:%M:%S")
    clm.type = "ROMS FRC file"

    # ---------- Dimensions ----------
    xi_rho = 602
    eta_rho = 402
    xi_u = 601
    eta_u = 402
    xi_v = 602
    eta_v = 401

    clm.createDimension('srf_time', ntime)
    clm.createDimension('eta_rho', eta_rho)
    clm.createDimension('xi_rho', xi_rho)

    clm.createDimension('shf_time', ntime)
    clm.createDimension('bhf_time', ntime)

    clm.createDimension('sms_time', ntime)
    clm.createDimension('eta_u', eta_u)
    clm.createDimension('xi_u', xi_u)
    clm.createDimension('eta_v', eta_v)
    clm.createDimension('xi_v', xi_v)

    # ---------- Time variables ----------
    def add_time_var(name):
        v = clm.createVariable(name, 'f8', (name,))
        # ✅ 按照 ocean_frc.nc 标准设置时间单位
        v.units = "seconds since 0001-01-01 00:00:00"
        v.calendar = "gregorian_proleptic"
        return v

    srf_time = add_time_var('srf_time')
    srf_time.long_name = "time for solar shortwave radiation flux"
    srf_time.field = "srf_time, scalar, series"

    shf_time = add_time_var('shf_time')
    shf_time.long_name = "time for surface net heat flux"
    shf_time.field = "shf_time, scalar, series"

    bhf_time = add_time_var('bhf_time')
    bhf_time.long_name = "time for bottom net heat flux"
    bhf_time.field = "bhf_time, scalar, series"

    sms_time = add_time_var('sms_time')
    sms_time.long_name = "time for surface u-momentum stress"
    sms_time.field = "sms_time, scalar, series"

    # ---------- Write Time Values ----------
    # 从 history 文件拷贝 ocean_time
    srf_time[:] = his.variables['ocean_time'][:]
    shf_time[:] = his.variables['ocean_time'][:]
    bhf_time[:] = his.variables['ocean_time'][:]
    sms_time[:] = his.variables['ocean_time'][:]

    # ---------- Forcing variables ----------
    def add_frc_var(name, dims):
        return clm.createVariable(name, 'f8', dims, zlib=True, complevel=1)

    # Solar Radiation
    swrad = add_frc_var('swrad', ('srf_time', 'eta_rho', 'xi_rho'))
    swrad.long_name = "solar shortwave radiation flux"
    swrad.units = "watt meter-2"
    swrad.field = "shortwave radiation, scalar, series"
    swrad.time = "srf_time"

    # Surface Heat Flux
    shflux = add_frc_var('shflux', ('shf_time', 'eta_rho', 'xi_rho'))
    shflux.long_name = "surface net heat flux"
    shflux.units = "watt meter-2"
    shflux.field = "surface heat flux, scalar, series"
    shflux.time = "shf_time"

    # Bottom Heat Flux
    bhflux = add_frc_var('bhflux', ('bhf_time', 'eta_rho', 'xi_rho'))
    bhflux.long_name = "bottom net heat flux"
    bhflux.units = "watt meter-2"
    bhflux.field = "bottom heat flux, scalar, series"
    bhflux.time = "bhf_time"

    # Surface U-Stress
    sustr = add_frc_var('sustr', ('sms_time', 'eta_u', 'xi_u'))
    sustr.long_name = "surface u-momentum stress"
    sustr.units = "newton meter-2"
    sustr.field = "surface u-momentum stress, scalar, series"
    sustr.time = "sms_time"

    # Surface V-Stress
    svstr = add_frc_var('svstr', ('sms_time', 'eta_v', 'xi_v'))
    svstr.long_name = "surface v-momentum stress"
    svstr.units = "newton meter-2"
    svstr.field = "surface v-momentum stress, scalar, series"
    svstr.time = "sms_time"

    # ---------- Fill all forcing variables with 0 ----------
    print(f"Filling forcing variables with zeros (shape: {ntime}, {eta_rho}, {xi_rho})...")

    # 使用切片赋值，因为数据量小（全是0），不会触发 HDF error
    # 对于大文件，如果想更安全，可以像之前那样用循环
    swrad[:, :, :] = 0.0
    shflux[:, :, :] = 0.0
    bhflux[:, :, :] = 0.0
    sustr[:, :, :] = 0.0
    svstr[:, :, :] = 0.0

    # ---------- Close files ----------
    clm.close()
    his.close()
    print("✅ FRC file created with zeros successfully")

if __name__ == '__main__':
    make_frc_shell(
        frc_file='/leader/user/zsm/tws_frc_offline.nc',
        his_file='/leader/user/zsm/TWS2_L1_ucla_new/TWS_L1_his_201903_full.nc'
    )
    
    
    
import netCDF4 as nc

def add_swflux_to_frc(frc_file):
    """
    在现有的 FRC 文件中增加 swflux 变量和 swf_time 时间变量。
    假设 FRC 文件已经存在，且包含 eta_rho, xi_rho, ocean_time 等维度。
    """
    # 1. 以追加模式打开 FRC 文件
    frc = nc.Dataset(frc_file, 'a')

    # 2. 获取时间维度（假设和 srf_time 或 shf_time 一致）
    # 优先使用 srf_time，如果没有则使用 ocean_time
    if 'srf_time' in frc.dimensions:
        time_dim = 'srf_time'
    elif 'shf_time' in frc.dimensions:
        time_dim = 'shf_time'
    else:
        # 兜底：使用第一个时间维度
        time_dim = list(frc.dimensions.keys())[0]
    
    ntime = frc.dimensions[time_dim].size
    eta_rho = frc.dimensions['eta_rho'].size
    xi_rho = frc.dimensions['xi_rho'].size

    # 3. 创建 swf_time 变量（如果不存在）
    if 'swf_time' not in frc.variables:
        swf_time = frc.createVariable('swf_time', 'd', (time_dim,))
        swf_time.long_name = "time for surface net fresh water flux"
        swf_time.units = "seconds since 2000-01-01 00:00:00"
        swf_time.calendar = "proleptic_gregorian"
        swf_time.field = "swf_time, scalar, series"
        
        # 复制时间值（从 srf_time 或 shf_time）
        if 'srf_time' in frc.variables:
            swf_time[:] = frc.variables['srf_time'][:]
        elif 'shf_time' in frc.variables:
            swf_time[:] = frc.variables['shf_time'][:]
        else:
            # 生成简单的时间序列
            swf_time[:] = range(ntime)

    # 4. 创建 swflux 变量（如果不存在）
    if 'swflux' not in frc.variables:
        swflux = frc.createVariable('swflux', 'f8', (time_dim, 'eta_rho', 'xi_rho'))
        swflux.long_name = "surface net fresh water flux"
        swflux.units = "centimeter day-1"  # ROMS 要求的单位
        swflux.field = "surface net fresh water flux, scalar, series"
        swflux.time = "swf_time"
        
        # 填充 0（或你自己的 E-P 数据）
        swflux[:, :, :] = 0.0

    # 5. 关闭文件
    frc.close()
    print("✅ swflux 和 swf_time 已成功添加到 FRC 文件")


# ===================== 使用示例 =====================
if __name__ == '__main__':
    frc_path = '/leader/user/zsm/tws_frc_offline.nc'
    add_swflux_to_frc(frc_path)

Filling forcing variables with zeros (shape: 744, 402, 602)...
✅ FRC file created with zeros successfully
✅ swflux 和 swf_time 已成功添加到 FRC 文件


In [11]:
frc = netCDF4.Dataset(
    '/leader/user/zsm/tws_frc_offline.nc',
    'a'   # 'a' = append / write
)

shflux_var1 = frc.variables['shflux']
swrad_var1 = frc.variables['swrad']
sustr_var1 = frc.variables['sustr']
svstr_var1 = frc.variables['svstr']
swflux_var1 = frc.variables['swflux']
ntime = shflux.shape[0]

shflux_var1[:, :, :] = shflux.astype(np.float64)
swrad_var1[:, :, :] = swrad.astype(np.float64)
sustr_var1[:, :, :] = sustr.astype(np.float64)
svstr_var1[:, :, :] = svstr.astype(np.float64)
swflux_var1[:, :, :] = swflux.astype(np.float64)

# for t in tqdm(range(ntime)):
#     if t % 50 == 0:
#         print(f"Writing shflux time {t}/{ntime}")

#     shflux_var1[t, :, :] = shflux[t, :, :].astype(np.float64)
#     swrad_var1[t, :, :] = swrad[t, :, :].astype(np.float64)
#     sustr_var1[t, :, :] = sustr[t, :, :].astype(np.float64)
#     svstr_var1[t, :, :] = svstr[t, :, :].astype(np.float64)
    



frc.close()

print("shflux, swrad,sustr,svstr written and FRC file saved successfully")

shflux, swrad,sustr,svstr written and FRC file saved successfully


In [43]:
sustr.shape,svstr.shape

((744, 402, 601), (744, 401, 602))

# create new grid

In [64]:
grd_i=xr.open_dataset('/sugon7/zsm/croco_tools_xmd1204/CROCO_FILES/TWS2/TWS2_L1_grid.nc')
grd_i

<xarray.Dataset> Size: 26MB
Dimensions:       (eta_rho: 402, xi_rho: 602, xi_u: 601, eta_v: 401,
                   eta_coarse: 202, xi_coarse: 302, s_rho: 45, s_w: 46)
Coordinates:
    lat_rho       (eta_rho, xi_rho) float64 2MB ...
    lon_rho       (eta_rho, xi_rho) float64 2MB ...
    lat_u         (eta_rho, xi_u) float64 2MB ...
    lon_u         (eta_rho, xi_u) float64 2MB ...
    lat_v         (eta_v, xi_rho) float64 2MB ...
    lon_v         (eta_v, xi_rho) float64 2MB ...
    lat_coarse    (eta_coarse, xi_coarse) float64 488kB ...
    lon_coarse    (eta_coarse, xi_coarse) float64 488kB ...
Dimensions without coordinates: eta_rho, xi_rho, xi_u, eta_v, eta_coarse,
                                xi_coarse, s_rho, s_w
Data variables: (12/15)
    angle         (eta_rho, xi_rho) float64 2MB ...
    f             (eta_rho, xi_rho) float64 2MB ...
    pm            (eta_rho, xi_rho) float64 2MB ...
    pn            (eta_rho, xi_rho) float64 2MB ...
    spherical     |S1 1B ...
    mask_rho      (eta_rho, xi_rho) int32 968kB ...
    ...            ...
    mask_coarse   (eta_coarse, xi_coarse) int32 244kB ...
    h             (eta_rho, xi_rho) float64 2MB ...
    sigma_r       (s_rho) float32 180B ...
    Cs_r          (s_rho) float32 180B ...
    sigma_w       (s_w) float32 184B ...
    Cs_w          (s_w) float32 184B ...
Attributes: (12/14)
    title:                   ROMS grid created by ROMS-Tools
    roms_tools_version:      3.0.1.dev7+gbba77f9
    size_x:                  300
    size_y:                  200
    center_lon:              119.3
    center_lat:              24.5
    ...                      ...
    topography_source_name:  SRTM15
    topography_source_path:  /leader/user/zsm/Dataset/Topo/SRTM15_V2.6.nc
    hmin:                    10.0
    theta_s:                 6.0
    theta_b:                 6.0
    hc:                      2.0

In [65]:
grd_i['angle']

<xarray.DataArray 'angle' (eta_rho: 402, xi_rho: 602)> Size: 2MB
[242004 values with dtype=float64]
Coordinates:
    lat_rho  (eta_rho, xi_rho) float64 2MB ...
    lon_rho  (eta_rho, xi_rho) float64 2MB ...
Dimensions without coordinates: eta_rho, xi_rho
Attributes:
    long_name:  Angle between xi axis and east
    units:      radians

In [66]:
def spheric_dist(lat1, lat2, lon1, lon2):
    """
    Compute the spherical distance between two points on Earth.
    
    Parameters:
    lat1, lat2 : array-like
        Latitude of the two points (in degrees).
    lon1, lon2 : array-like
        Longitude of the two points (in degrees).
    
    Returns:
    dist : array-like
        The spherical distance between the points (in meters).
    """
    
    # Earth radius in meters
    R = 6367442.76
    
    # Determine proper longitudinal shift
    l = np.abs(lon2 - lon1)
    l[l >= 180] = 360 - l[l >= 180]
    
    # Convert decimal degrees to radians
    deg2rad = np.pi / 180
    lat1 = lat1 * deg2rad
    lat2 = lat2 * deg2rad
    l = l * deg2rad
    
    # Compute the distances
    dist = R * np.arcsin(np.sqrt(((np.sin(l) * np.cos(lat2)) ** 2) + 
                                 ((np.sin(lat2) * np.cos(lat1)) - 
                                  (np.sin(lat1) * np.cos(lat2) * np.cos(l))) ** 2))
    
    return dist

import netCDF4
import numpy as np

import netCDF4
import numpy as np

def spheric_dist(lat1, lat2, lon1, lon2):
    R = 6378137.0
    lat1 = np.deg2rad(lat1)
    lat2 = np.deg2rad(lat2)
    lon1 = np.deg2rad(lon1)
    lon2 = np.deg2rad(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

def get_metrics(grdname):
    nc = netCDF4.Dataset(grdname, 'r')

    latu = nc.variables['lat_u'][:]
    lonu = nc.variables['lon_u'][:]
    latv = nc.variables['lat_v'][:]
    lonv = nc.variables['lon_v'][:]

    nc.close()

    Mp, L = latu.shape
    M, Lp = latv.shape

    Lm = L - 1
    Mm = M - 1

    # --------- dx ---------
    dx = np.zeros((Mp, Lp))

    dx[:, 1:L] = spheric_dist(
        latu[:, 0:Lm], latu[:, 1:L],
        lonu[:, 0:Lm], lonu[:, 1:L]
    )

    dx[:, 0] = dx[:, 1]
    dx[:, Lp-1] = dx[:, Lm]

    # --------- dy ---------
    dy = np.zeros((Mp, Lp))

    dy[1:M, :] = spheric_dist(
        latv[0:Mm, :], latv[1:M, :],
        lonv[0:Mm, :], lonv[1:M, :]
    )

    dy[0, :] = dy[1, :]
    dy[Mp-1, :] = dy[Mm, :]

    # --------- pm, pn ---------
    pm = 1.0 / dx
    pn = 1.0 / dy

    # --------- dndx, dmde ---------
    dndx = np.zeros_like(pm)
    dmde = np.zeros_like(pm)

    dndx[1:M, 1:L] = 0.5 * (
        1.0 / pn[1:M, 2:L+1] - 1.0 / pn[1:M, 0:L-1]
    )

    dmde[1:M, 1:L] = 0.5 * (
        1.0 / pm[2:M+1, 1:L] - 1.0 / pm[0:M-1, 1:L]
    )

    # --------- 边界清零 ---------
    dndx[0, :] = 0
    dndx[Mp-1, :] = 0
    dndx[:, 0] = 0
    dndx[:, Lp-1] = 0

    dmde[0, :] = 0
    dmde[Mp-1, :] = 0
    dmde[:, 0] = 0
    dmde[:, Lp-1] = 0

    return pm, pn, dndx, dmde


def rho2uvp(rfield):
    """
    Convert rho-point field to u, v, psi points
    Equivalent to MATLAB rho2uvp.m
    """

    Mp, Lp = rfield.shape
    M = Mp - 1
    L = Lp - 1

    # ---------- vfield (eta direction average) ----------
    vfield = 0.5 * (rfield[0:M, :] + rfield[1:Mp, :])

    # ---------- ufield (xi direction average) ----------
    ufield = 0.5 * (rfield[:, 0:L] + rfield[:, 1:Lp])

    # ---------- pfield (eta direction average of ufield) ----------
    pfield = 0.5 * (ufield[0:M, :] + ufield[1:Mp, :])

    return ufield, vfield, pfield

def rad2deg(angle_rad):
    """
    Convert angle from radians to degrees
    """
    return angle_rad * 180.0 / np.pi

In [67]:
pm1, pn1, dndx, dmde=get_metrics('/sugon7/zsm/croco_tools_xmd1204/CROCO_FILES/TWS2/TWS2_L1_grid.nc')


In [68]:
lonr=grd_i['lon_rho'].values
latr=grd_i['lat_rho'].values

lonu=grd_i['lon_u'].values
lonv=grd_i['lon_v'].values

latu=grd_i['lat_u'].values
latv=grd_i['lat_v'].values

maskr=grd_i['mask_rho'].values
masku=grd_i['mask_u'].values
maskv=grd_i['mask_v'].values

pm=grd_i['pm'].values
pn=grd_i['pn'].values

h=grd_i['h'].values

f=grd_i['f'].values

_,_,lonp=rho2uvp(lonr)
_,_,latp=rho2uvp(latr)
_,_,maskp=rho2uvp(maskr)

angle=rad2deg(grd_i['angle'].values)

xl=np.mean(np.sum(1/pm,axis=1))

el=np.mean(np.sum(1/pn,axis=0))


In [69]:
# maskr[310:340,400:440]=0
maskr[h<12]=0

In [70]:
import netCDF4
import numpy as np

def make_roms_grid_shell(grid_file):

    # 使用 NETCDF3_CLASSIC，这是 ROMS 最兼容的格式
    grd = netCDF4.Dataset(grid_file, 'w', format='NETCDF3_CLASSIC')

    # ---------- Global attributes ----------
    grd.type = "ROMS GRID file"
    grd.gridid = "theGridTitle"
    grd.title = "ROMS Application"
    grd.history = "Created by Python script, on 2026-04-16"

    # ---------- Dimensions ----------
    grd.createDimension('xi_psi', 601)
    grd.createDimension('xi_rho', 602)
    grd.createDimension('xi_u', 601)
    grd.createDimension('xi_v', 602)

    grd.createDimension('eta_psi', 401)
    grd.createDimension('eta_rho', 402)
    grd.createDimension('eta_u', 402)
    grd.createDimension('eta_v', 401)

    grd.createDimension('one', 1)
    grd.createDimension('two', 2)
    grd.createDimension('bath', 1)

    # ---------- Variables ----------

    # Domain size
    xl = grd.createVariable('xl', 'f8', ('one',))
    xl.long_name = "domain length in the XI-direction"
    xl.units = "meter"

    el = grd.createVariable('el', 'f8', ('one',))
    el.long_name = "domain length in the ETA-direction"
    el.units = "meter"

    # Projection
    JPRJ = grd.createVariable('JPRJ', 'c', ('two',))
    JPRJ.long_name = "Map projection type"
    JPRJ.option_ME = "Mercator"
    JPRJ.option_ST = "Stereographic"
    JPRJ.option_LC = "Lambert conformal conic"

    # Spherical switch
    spherical = grd.createVariable('spherical', 'c', ('one',))
    spherical.long_name = "Grid type logical switch"
    spherical.option_T = "spherical"
    spherical.option_F = "Cartesian"
    spherical[:] = "T"

    # Depth limits
    depthmin = grd.createVariable('depthmin', 'i2', ('one',))
    depthmin.long_name = "domain length in the XI-direction"
    depthmin.units = "meter"

    depthmax = grd.createVariable('depthmax', 'i2', ('one',))
    depthmax.long_name = "Deep bathymetry clipping depth"
    depthmax.units = "meter"

    # Bathymetry
    hraw = grd.createVariable('hraw', 'f8', ('bath', 'eta_rho', 'xi_rho'))
    hraw.long_name = "Working bathymetry at RHO-points"
    hraw.units = "meter"
    hraw.field = "bath, scalar"

    h = grd.createVariable('h', 'f8', ('eta_rho', 'xi_rho'))
    h.long_name = "Final bathymetry at RHO-points"
    h.units = "meter"
    h.field = "bath, scalar"

    # Coriolis
    f = grd.createVariable('f', 'f8', ('eta_rho', 'xi_rho'))
    f.long_name = "Coriolis parameter at RHO-points"
    f.units = "second-1"
    f.field = "Coriolis, scalar"

    # Metrics
    pm = grd.createVariable('pm', 'f8', ('eta_rho', 'xi_rho'))
    pm.long_name = "curvilinear coordinate metric in XI"
    pm.units = "meter-1"
    pm.field = "pm, scalar"

    pn = grd.createVariable('pn', 'f8', ('eta_rho', 'xi_rho'))
    pn.long_name = "curvilinear coordinate metric in ETA"
    pn.units = "meter-1"
    pn.field = "pn, scalar"

    dndx = grd.createVariable('dndx', 'f8', ('eta_rho', 'xi_rho'))
    dndx.long_name = "xi derivative of inverse metric factor pn"
    dndx.units = "meter"
    dndx.field = "dndx, scalar"

    dmde = grd.createVariable('dmde', 'f8', ('eta_rho', 'xi_rho'))
    dmde.long_name = "eta derivative of inverse metric factor pm"
    dmde.units = "meter"
    dmde.field = "dmde, scalar"

    # Cartesian coordinates
    x_rho = grd.createVariable('x_rho', 'f8', ('eta_rho', 'xi_rho'))
    x_rho.long_name = "x location of RHO-points"
    x_rho.units = "meter"

    y_rho = grd.createVariable('y_rho', 'f8', ('eta_rho', 'xi_rho'))
    y_rho.long_name = "y location of RHO-points"
    y_rho.units = "meter"

    x_psi = grd.createVariable('x_psi', 'f8', ('eta_psi', 'xi_psi'))
    x_psi.long_name = "x location of PSI-points"
    x_psi.units = "meter"

    y_psi = grd.createVariable('y_psi', 'f8', ('eta_psi', 'xi_psi'))
    y_psi.long_name = "y location of PSI-points"
    y_psi.units = "meter"

    x_u = grd.createVariable('x_u', 'f8', ('eta_u', 'xi_u'))
    x_u.long_name = "x location of U-points"
    x_u.units = "meter"

    y_u = grd.createVariable('y_u', 'f8', ('eta_u', 'xi_u'))
    y_u.long_name = "y location of U-points"
    y_u.units = "meter"

    x_v = grd.createVariable('x_v', 'f8', ('eta_v', 'xi_v'))
    x_v.long_name = "x location of V-points"
    x_v.units = "meter"

    y_v = grd.createVariable('y_v', 'f8', ('eta_v', 'xi_v'))
    y_v.long_name = "y location of V-points"
    y_v.units = "meter"

    # Latitude / Longitude
    lat_rho = grd.createVariable('lat_rho', 'f8', ('eta_rho', 'xi_rho'))
    lat_rho.long_name = "latitude of RHO-points"
    lat_rho.units = "degree_north"

    lon_rho = grd.createVariable('lon_rho', 'f8', ('eta_rho', 'xi_rho'))
    lon_rho.long_name = "longitude of RHO-points"
    lon_rho.units = "degree_east"

    lat_psi = grd.createVariable('lat_psi', 'f8', ('eta_psi', 'xi_psi'))
    lat_psi.long_name = "latitude of PSI-points"
    lat_psi.units = "degree_north"

    lon_psi = grd.createVariable('lon_psi', 'f8', ('eta_psi', 'xi_psi'))
    lon_psi.long_name = "longitude of PSI-points"
    lon_psi.units = "degree_east"

    lat_u = grd.createVariable('lat_u', 'f8', ('eta_u', 'xi_u'))
    lat_u.long_name = "latitude of U-points"
    lat_u.units = "degree_north"

    lon_u = grd.createVariable('lon_u', 'f8', ('eta_u', 'xi_u'))
    lon_u.long_name = "longitude of U-points"
    lon_u.units = "degree_east"

    lat_v = grd.createVariable('lat_v', 'f8', ('eta_v', 'xi_v'))
    lat_v.long_name = "latitude of V-points"
    lat_v.units = "degree_north"

    lon_v = grd.createVariable('lon_v', 'f8', ('eta_v', 'xi_v'))
    lon_v.long_name = "longitude of V-points"
    lon_v.units = "degree_east"

    # Masks
    mask_rho = grd.createVariable('mask_rho', 'f8', ('eta_rho', 'xi_rho'))
    mask_rho.long_name = "mask on RHO-points"
    mask_rho.option_0 = "land"
    mask_rho.option_1 = "water"

    mask_u = grd.createVariable('mask_u', 'f8', ('eta_u', 'xi_u'))
    mask_u.long_name = "mask on U-points"
    mask_u.option_0 = "land"
    mask_u.option_1 = "water"

    mask_v = grd.createVariable('mask_v', 'f8', ('eta_v', 'xi_v'))
    mask_v.long_name = "mask on V-points"
    mask_v.option_0 = "land"
    mask_v.option_1 = "water"

    mask_psi = grd.createVariable('mask_psi', 'f8', ('eta_psi', 'xi_psi'))
    mask_psi.long_name = "mask on PSI-points"
    mask_psi.option_0 = "land"
    mask_psi.option_1 = "water"

    # Angle
    angle = grd.createVariable('angle', 'f8', ('eta_rho', 'xi_rho'))
    angle.long_name = "angle between xi axis and east"
    angle.units = "degree"

    # ---------- 不写入数据（只造壳子） ----------
    grd.close()
    print(f"✅ ROMS Grid shell created: {grid_file}")

# 运行创建
make_roms_grid_shell('/leader/user/zsm/TWS_L1_grd_offline_test.nc')

✅ ROMS Grid shell created: /leader/user/zsm/TWS_L1_grd_offline_test.nc


In [71]:
grd0 = netCDF4.Dataset(
    '/leader/user/zsm/TWS_L1_grd_offline_test.nc',
    'a'   # 'a' = append / write
)
grd0

<class 'netCDF4.Dataset'>
root group (NETCDF3_CLASSIC data model, file format NETCDF3):
    type: ROMS GRID file
    gridid: theGridTitle
    title: ROMS Application
    history: Created by Python script, on 2026-04-16
    dimensions(sizes): xi_psi(601), xi_rho(602), xi_u(601), xi_v(602), eta_psi(401), eta_rho(402), eta_u(402), eta_v(401), one(1), two(2), bath(1)
    variables(dimensions): float64 xl(one), float64 el(one), |S1 JPRJ(two), |S1 spherical(one), int16 depthmin(one), int16 depthmax(one), float64 hraw(bath, eta_rho, xi_rho), float64 h(eta_rho, xi_rho), float64 f(eta_rho, xi_rho), float64 pm(eta_rho, xi_rho), float64 pn(eta_rho, xi_rho), float64 dndx(eta_rho, xi_rho), float64 dmde(eta_rho, xi_rho), float64 x_rho(eta_rho, xi_rho), float64 y_rho(eta_rho, xi_rho), float64 x_psi(eta_psi, xi_psi), float64 y_psi(eta_psi, xi_psi), float64 x_u(eta_u, xi_u), float64 y_u(eta_u, xi_u), float64 x_v(eta_v, xi_v), float64 y_v(eta_v, xi_v), float64 lat_rho(eta_rho, xi_rho), float64 lon_rho(e

In [72]:
# grd0.close()

In [73]:
LON_R = grd0.variables['lon_rho']
LAT_R = grd0.variables['lat_rho']

LON_U = grd0.variables['lon_u']
LON_V = grd0.variables['lon_v']
LON_P = grd0.variables['lon_psi']

LAT_U = grd0.variables['lat_u']
LAT_V = grd0.variables['lat_v']
LAT_P = grd0.variables['lat_psi']

M_U = grd0.variables['mask_u']
M_V = grd0.variables['mask_v']
M_P = grd0.variables['mask_psi']
M_R = grd0.variables['mask_rho']

PM = grd0.variables['pm']
PN = grd0.variables['pn']
H = grd0.variables['h']
F = grd0.variables['f']
ANG = grd0.variables['angle']

XL = grd0.variables['xl']
EL = grd0.variables['el']

#---------------------------------------------------------------------------------------------------------
LON_R[:, :] = lonr.astype(np.float64)
LAT_R[:, :] = latr.astype(np.float64)

LON_U[:, :] = lonu.astype(np.float64)
LON_V[:, :] = lonv.astype(np.float64)
LON_P[:, :] = lonp.astype(np.float64)

LAT_U[:, :] = latu.astype(np.float64)
LAT_V[:, :] = latv.astype(np.float64)
LAT_P[:, :] = latp.astype(np.float64)

M_U[:, :] = masku.astype(np.float64)
M_V[:, :] = maskv.astype(np.float64)
M_P[:, :] = maskp.astype(np.float64)
M_R[:, :] = maskr.astype(np.float64)

PM[:, :] = pm.astype(np.float64)
PN[:, :] = pn.astype(np.float64)

H[:, :] = h.astype(np.float64)
F[:, :] = f.astype(np.float64)
ANG[:, :] = angle.astype(np.float64)

XL[:] = xl.astype(np.float64)
EL[:] = el.astype(np.float64)
grd0.close()

print('processing grd file done!')

processing grd file done!


In [74]:
# h.min()